**Streaming GridFM data tutorial**
=================================

This tutorial aims at streaming a dataset on which one can use GridFM datamodules and datasets. This allows to use energydiff on one node and adds the possibility to use a temperature covariable.

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../../data")

Streaming electricity data
--------------------------

We use Hydro-Québec open data for this example. Here it corresponds to 2019-2024 electricity consumption in the whole Québec.

We can replace this dataset by any dataset of power or energy target of one node.

In [2]:
from opensynth.datasets import datasets_utils

URL = "https://donnees.hydroquebec.com/api/explore/v2.1/catalog/datasets/historique-demande-electricite-quebec/exports/csv?lang=fr&timezone=America%2FToronto&use_labels=true&delimiter=%2C" 
FILE_NAME = Path(f"{data_dir}/raw/hq_data_2019-2024.csv")  
datasets_utils.download_data(URL, FILE_NAME)

Streaming temperature data
--------------------------

We use the API env_canada to get temperature values of Québec. We use a weather station in Montréal since electricity consumption is heavier in the South of Québec. This example is formated with daily mean temperature but we could change it to hourly temperature if the covariable is coded accordingly in the model.

Again we could replace temperature data by any temperature data as long as it matches the dates and resolution of the target data above.

In [3]:
import nest_asyncio
from env_canada import ECHistoricalRange
import datetime as dt

nest_asyncio.apply()

weather = ECHistoricalRange(station_id=7025251, timeframe='hourly', daterange=(dt.datetime(2019, 1, 1), dt.datetime(2024, 12, 31)),)

weather = weather.get_data()

weather = weather.rename(columns={'Temp (°C)': 'temperature'})
temperature = weather['temperature']
temperature.index.name = 'datetime'

temperature.to_csv(f'{data_dir}/raw/temperature_2019-2024.csv')

Preprocessing data
------------------
We first merge datasets and then preprocess them.

In [4]:
df = pd.read_csv(f'{data_dir}/raw/hq_data_2019-2024.csv')
# Formating the date column to datetime and renaming columns for clarity
df['date'] = pd.to_datetime(df["date"].apply(lambda x: x[:19]))
df = df.rename(columns={'date': 'datetime', 'demande (MW)': 'demand'})
df['date'] = pd.to_datetime(df['datetime'].dt.date)
df['time'] = df['datetime'].dt.time

# Adding an id colunm even for one node so the data can be used in the same way as the multi-node data
df['ID'] = 'quebec'

# Merging the temperature data with the demand data on the date column
out = pd.merge(temperature, df, how='inner', on='datetime')
out.to_csv(f'{data_dir}/raw/gridfm_data_example.csv', index=False)


In [5]:
from opensynth.datasets.gridfm import get_data

#Preprocessing the data (same as in app.py)
get_data.split_preprocess_data(
        split=True,
        preprocess=True,
        data_dir=data_dir,
        csv_data_path=f'/raw/gridfm_data_example.csv',
        sample_fraction=0.75,
        time_resolution='hourly',
        feature_cols=['temperature'],
        id_col='ID',
        kwh_col='demand',
        datetime_col='datetime',
        utc=False,
        datetime_format=None,
        historical_start='2019-01-01',
        historical_end='2024-12-31',
        future_start='2025-01-01',
        future_end='2025-01-01',
        drop_nulls=True,
    )